# Deep Learning 006 — Loss Functions

Lesson 005 ended with an algorithm that had a *rule* but no *goal*. A loss function is
the goal. Here we write the perceptron's loss down explicitly, descend it, and then
change two lines to get logistic regression out of the same code.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

df = pd.read_csv('../data/placement.csv')
X = df[['cgpa', 'resume_score']].to_numpy()
y = df['placed'].to_numpy().astype(int)
Xb = np.c_[np.ones(len(X)), X]        # bias folded in as column 0
t = 2 * y - 1                         # labels as -1 / +1

## The perceptron loss

$$L = \frac{1}{n}\sum_i \max(0,\; -t_i \cdot (w \cdot x_i))$$

Zero for a correctly classified point, and growing linearly with how wrong a
misclassified one is. **The gradient is what produces the update rule of lesson 005** —
the "trick" was gradient descent all along.

In [ ]:
def perceptron_loss(w, Xb, t):
    return np.maximum(0, -t * (Xb @ w)).mean()

def perceptron_grad(w, Xb, t):
    margin = t * (Xb @ w)
    active = (margin <= 0).astype(float)      # only mistakes contribute
    return -(Xb * (active * t)[:, None]).mean(0)

def descend(w0, loss_fn, grad_fn, lr=0.1, steps=400):
    w = w0.copy()
    curve = []
    for _ in range(steps):
        curve.append(loss_fn(w, Xb, t if loss_fn is perceptron_loss else y))
        w -= lr * grad_fn(w, Xb, t if grad_fn is perceptron_grad else y)
    return w, np.array(curve)

w0 = np.zeros(3)
wp, curve_p = descend(w0, perceptron_loss, perceptron_grad, lr=0.05, steps=600)
acc_p = ((Xb @ wp >= 0).astype(int) == y).mean()
print(f'perceptron loss: final {curve_p[-1]:.4f}   accuracy {acc_p:.1%}')

## Now change two things and you have logistic regression

Swap `step` for `sigmoid`, and the hinge for **log loss**:

$$L = -\frac{1}{n}\sum_i \big[y_i\log \hat y_i + (1-y_i)\log(1-\hat y_i)\big]$$

The gradient comes out *simpler* than the perceptron's, which is one of the nicer
surprises in this course.

In [ ]:
def sigmoid(z):
    return np.where(z >= 0, 1 / (1 + np.exp(-np.abs(z))),
                    np.exp(-np.abs(z)) / (1 + np.exp(-np.abs(z))))

def log_loss(w, Xb, y):
    p = np.clip(sigmoid(Xb @ w), 1e-12, 1 - 1e-12)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p)).mean()

def log_grad(w, Xb, y):
    return Xb.T @ (sigmoid(Xb @ w) - y) / len(y)     # that is all of it

wl, curve_l = descend(w0, log_loss, log_grad, lr=0.5, steps=600)
acc_l = ((sigmoid(Xb @ wl) >= 0.5).astype(int) == y).mean()
print(f'log loss:        final {curve_l[-1]:.4f}   accuracy {acc_l:.1%}')

sk = LogisticRegression(penalty=None, max_iter=5000).fit(X, y)
print(f'sklearn LogisticRegression        accuracy {sk.score(X, y):.1%}')
print(f'\nour w  {np.round(wl, 3)}')
print(f'sklearn w  {np.round(np.r_[sk.intercept_, sk.coef_.ravel()], 3)}')
print('\nSame direction, different scale - log loss keeps pushing the weights')
print('larger to sharpen confidence, so the magnitude depends on when you stop.')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(curve_p, label='perceptron loss')
ax[0].plot(curve_l, label='log loss')
ax[0].set(xlabel='step', ylabel='loss', title='both descend')
ax[0].legend()

z = np.linspace(-3, 3, 300)
ax[1].plot(z, np.maximum(0, -z), label='perceptron: max(0, -t z)')
ax[1].plot(z, np.log(1 + np.exp(-z)), label='log loss (t=+1)')
ax[1].axvline(0, color='k', lw=0.6)
ax[1].set(xlabel='t * (w . x)', ylabel='penalty',
          title='the shapes: note log loss is never exactly 0')
ax[1].legend(fontsize=8); plt.tight_layout(); plt.show()

**Look at the right-hand plot — it is the whole difference.** The perceptron loss is
*flat at zero* for every correctly classified point, so those points stop contributing
and the line stops improving once it is merely correct. Log loss is **never exactly
zero**: a correct-but-barely point still pushes. That is why logistic regression settles
on a confident boundary while the perceptron settles for any boundary that works — and
why, on non-separable data, log loss converges while the perceptron oscillates.

## Exercises

1. Plot the decision boundaries of `wp` and `wl` together. Which looks better placed,
   and does the accuracy agree?
2. Run the perceptron-loss descent on the separable data from lesson 005. Does the loss
   reach exactly 0? Does the log loss?
3. Implement **hinge loss** `max(0, 1 - t·z)` (an SVM's loss) and add it to the
   comparison. Where does it sit between the two?